# FastConv: versao com acumulador em s

Este notebook contem o segundo exemplo movido: cada linha de `S` e calculada uma vez, gera uma linha de `S' = S A`, e sua contribuicao e acumulada em `s = A^T S'`.

In [ ]:
import numpy as np
from fractions import Fraction

C = np.array([
    [-1, 0,  0,  0],
    [ 0, 1, -1, -1],
    [ 1, 1,  1,  0],
    [ 0, 0,  0,  1],
], dtype=int)

d = np.array([
    [ 0,  1,  2,  3],
    [ 4,  5,  6,  7],
    [ 8,  9, 10, 11],
    [12, 13, 14, 15],
], dtype=int)

G = np.array([
    [Fraction(0),     Fraction(-3, 2), Fraction(-1, 2), Fraction(-2)],
    [Fraction(-9, 2), Fraction(9),     Fraction(3),     Fraction(15, 2)],
    [Fraction(-3, 2), Fraction(3),     Fraction(1),     Fraction(5, 2)],
    [Fraction(-6),    Fraction(21, 2), Fraction(7, 2),  Fraction(8)],
], dtype=object)

A = np.array([
    [1,  0],
    [1,  1],
    [1, -1],
    [0,  1],
], dtype=object)

def dot_expression(a, b):
    return " + ".join(f"({x})*({y})" for x, y in zip(a, b))


In [ ]:
def compute_D_row(row):
    c_t_row = C.T[row, :]
    D_prime_row = c_t_row @ d
    D_row = D_prime_row @ C

    print(f"Linha {row} de C^T:")
    print(c_t_row)
    print(f"\n1) Calculando D'[{row}, :] = C^T[{row}, :] d")
    for col in range(d.shape[1]):
        expr = dot_expression(c_t_row, d[:, col])
        print(f"D'[{row}, {col}] = {expr} = {D_prime_row[col]}")

    print(f"\nD'[{row}, :] = {D_prime_row.tolist()}")
    print(f"\n2) Calculando D[{row}, :] = D'[{row}, :] C")
    for col in range(C.shape[1]):
        expr = dot_expression(D_prime_row, C[:, col])
        print(f"D[{row}, {col}] = {expr} = {D_row[col]}")

    print(f"\nD[{row}, :] = {D_row.tolist()}")
    return D_row

def compute_S_row(row):
    D_row = compute_D_row(row)
    S_row = np.zeros(4, dtype=object)

    print(f"\n3) Calculando S[{row}, :] = D[{row}, :] # G[{row}, :]")
    for col in range(D_row.shape[0]):
        S_row[col] = D_row[col] * G[row, col]
        print(f"S[{row}, {col}] = D[{row}, {col}] * G[{row}, {col}] = ({D_row[col]})*({G[row, col]}) = {S_row[col]}")

    print(f"\nS[{row}, :] = {S_row.tolist()}")
    return S_row


In [ ]:
s_accum = np.zeros((2, 2), dtype=object)
S_prime_rows = []

for row in range(C.shape[0]):
    print("=" * 70)
    print(f"Fluxo da linha {row}")

    S_row = compute_S_row(row)
    S_prime_row = S_row @ A
    S_prime_rows.append(S_prime_row)

    print(f"\n4) Calculando S'[{row}, :] = S[{row}, :] A")
    for col in range(A.shape[1]):
        expr = dot_expression(S_row, A[:, col])
        print(f"S'[{row}, {col}] = {expr} = {S_prime_row[col]}")

    print(f"\n5) Acumulando A[{row}, out_row] * S'[{row}, :] em s[out_row, :]")
    for out_row in range(A.shape[1]):
        contribution = A[row, out_row] * S_prime_row
        s_accum[out_row, :] += contribution
        print(f"s[{out_row}, :] += A[{row}, {out_row}] * S'[{row}, :] = ({A[row, out_row]}) * {S_prime_row.tolist()} = {contribution.tolist()}")
        print(f"s[{out_row}, :] parcial = {s_accum[out_row, :].tolist()}")

    print()


In [ ]:
S_prime_accum = np.vstack(S_prime_rows)
S_reference = (C.T @ d @ C).astype(object) * G

print("S_prime_accum = S A =")
print(S_prime_accum)
print("\ns_accum = A^T S_prime_accum = A^T S A =")
print(s_accum)

assert np.array_equal(S_prime_accum, S_reference @ A)
assert np.array_equal(s_accum, A.T @ S_reference @ A)
